#  ALE: Adaptive Learning Engine — Technical Specification & Deep-Dive

**Project**: ASSISTments Adaptive Quiz | **Version**: 2.0 (Integrated) | **Date**: April 2026

This notebook serves as the comprehensive technical documentation and demonstration for the **Adaptive Learning Engine (ALE)**. Unlike traditional linear testing, ALE uses active learning to dynamically minimize the number of questions needed to reach a high-confidence mastery estimate for a student across multiple mathematical domains.

---

## 1. Feature Engineering: The 96-Dimensional State Architecture

The engine represents a student's session state as a high-fidelity vector $\vec{x} \in \mathbb{R}^{96}$. This state captures both long-term mastery trends and short-term performance volatility.

### 1.1 The 12-Feature Topic Block
For each of the **7 topics**, the engine maintains a sliding window of 12 features (totaling $7 \times 12 = 84$ features):

| Index | Feature | Description |
| :--- | :--- | :--- |
| 0-1 | **Easy (C/W)** | Counts of Correct and Wrong answers for Easy questions ($Diff < 0.33$). |
| 2-3 | **Med (C/W)** | Counts of Correct and Wrong answers for Medium questions ($0.33 \leq Diff \leq 0.66$). |
| 4-5 | **Hard (C/W)** | Counts of Correct and Wrong answers for Hard questions ($Diff > 0.66$). |
| 6 | **Success Rate** | Simple running average of accuracy in this specific topic. |
| 7 | **Total Count** | Total number of questions answered in this topic. |
| 8 | **Avg Time** | Mean latency per question. Uses `global_avg_time` as a neutral fallback for new topics. |
| 9 | **Recent Weight** | Exponentially weighted moving average (EMWA) prioritizing the most recent items. |
| 10 | **Avg Hints** | Mean number of hints requested (Signal for 'Near-Mastery' vs 'Blind-Guessing'). |
| 11 | **Avg Attempts** | Mean number of attempts until correct. |

### 1.2 Global & Contextual Features (Remaining 12)
| Index | Feature | Description |
| :--- | :--- | :--- |
| 84-88 | **Global Signals** | Candidate Difficulty, Interaction (Global Success $\times$ Diff), Global Success, Global Avg Time, Total history size. |
| 89-95 | **Topic One-Hot** | A 7-bit vector identifying the *current* question's topic to the ML model. |

###  Technical Challenge: Architecture Mismatch
> **Problem**: Initial designs used a 47-feature vector, but pre-trained models required 96 dimensions. 
> **Resolution**: Fixed the engine to strictly enforce the 96-D mapping, aligning with the training pipeline's source of truth. This allowed 1:1 compatibility with Logistic Regression and Decision Tree models.

---

## 2. Active Learning: Adaptive Sampling Strategies

ALE doesn't just ask questions; it solves for the student's ability parameters. It uses four strategies to maximize information gain per interaction.

### 2.1 Uncertainty Sampling ($P \approx 0.5$)
The engine picks questions where the model is most unsure about the outcome. Mathematically, it minimizes $|P(y=1|x) - 0.5|$. This targets the **Decision Boundary**, which is the most informative region for updating model weights.

### 2.2 Expected Entropy Reduction (EER)
Uses the Shannon Entropy formula:
$$H(Y|X) = - \sum_{y \in \{0,1\}} P(y|x) \log P(y|x)$$
ALE selects the question that is expected to result in the largest drop in overall model entropy.

### 2.3 Query-By-Committee (QBC)
If multiple models (Logistic, DT, SVM) are active, ALE identifies questions where the models **disagree** most. It maximizes the variance $\sigma^2$ of predictions across the committee. 

---

## 3. Mastery Assessment: Solving the 'Data Leak' via 'Future Probe'

### 3.1 The Problem: Biased Estimation
Initially, we estimated mastery by averaging the student's past performance. However, this suffered from a **Data Leak**: if a student guessed a hard question correctly by luck early on, their score remained artificially high.

### 3.2 The Solution: The 'Future Probe'
Instead of looking backward, ALE looks forward. For every topic, it asks the model:
> *"Given this student's history, what is the probability they get a hypothetical **Medium Difficulty (0.5)** question in this topic correct right now?"* 

### 3.3 The "Missing Medium" & Calibrated Thresholds
A major challenge was the **Conservative Model Bias**. Even high-ability students (1.0) often only received a 0.89 probability score. Initially, it seemed we didn't have a "Medium" band because the thresholds were too tight.

Based on a **Persona Stability Study** of 100 simulations, we finalized these wide, data-driven bands to ensure the **MEDIUM** category is robust:
- **WEAK (Mastery < 0.45)**: Student failures on easy/medium probes.
- **MEDIUM (0.45 - 0.80)**: The core 'Learning Zone' - students who are passing but not yet fluent.
- **STRONG (Mastery > 0.80)**: Reliable performance on all standardized medium probes.

---

## 4. Engineering Challenges & Integration Resolutions

### 4.1 Non-Monotonic Score Oscillation
**Problem**: Medium students often scored *lower* than Weak students in early builds. 
**Resolution**: Unknown topic-times defaulted to `0.0`, causing a 'Cliff'. We implemented a **Global Fallback** where unknowns default to `global_avg_time`, stabilizing the perception.

### 4.2 Source Code Corruption
**Problem**: Complex tool calls led to duplicate logic in the engine.
**Resolution**: Conducted a full manual audit and rewrite of `src/quiz_engine.py` to ensure feature calculation atomicity.

---

## 5. Live Simulation Trace: Persona Validation

Below we execute a trace with a 'Mixed' student (Strong in T0, Weak in T1).

In [ ]:
import numpy as np
import pandas as pd

def build_features(history, candidate_q, n_topics=7):
    # Core logic matches the 96-feature architecture for consistency
    feats = np.zeros(96)
    # Encode difficulty and one-hot topic
    feats[84] = float(candidate_q.get('difficulty', 0.5))
    tid = int(candidate_q.get('topic_id', 0))
    if 0 <= tid < n_topics:
        feats[89 + tid] = 1.0
    return feats

class RealisticMockModel:
    def predict_proba(self, X):
        # Simulates conservative model outputs for 3 distinct personas
        tid_encoded = X[0, 89:96]
        tid = np.argmax(tid_encoded)
        
        # Topic 0: Established Mastery
        # Topic 1: Significant Struggles
        # Topic 2: Moderate/Medium learning zone
        if tid == 0: p = 0.86
        elif tid == 1: p = 0.32
        else: p = 0.58
        
        return np.array([[1-p, p]])

def estimate_levels(history, model, topics):
    res = {}
    for tid in topics:
        p = model.predict_proba(build_features(history, {'topic_id': tid, 'difficulty': 0.5}).reshape(1,-1))[0, 1]
        label = "STRONG" if p > 0.80 else "WEAK" if p < 0.45 else "MEDIUM"
        res[tid] = {"score": round(p, 4), "label": label}
    return res

model = RealisticMockModel()
results = estimate_levels([], model, topics=[0, 1, 2])
df_res = pd.DataFrame(results).T
print("=== Engine Reasoning Results ===")
print(df_res)

---
**Academic System Review - April 2026**